In [3]:
!pip -q install speechbrain asteroid torchaudio soundfile
import torch, torchaudio, glob, random, os
print(torch.__version__, "| CUDA:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.4/156.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 41.6 MB/s eta 0:00:00
2.10.0+cu128 | CUDA: True


In [4]:
!wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz
!tar -xzf dev-clean.tar.gz
flacs = glob.glob("LibriSpeech/dev-clean/**/*.flac", recursive=True)
by_spk = {}
for f in flacs:
    by_spk.setdefault(f.split("/")[-3], []).append(f)
print(len(flacs), "files |", len(by_spk), "speakers")   

2703 files | 40 speakers


In [5]:
SR = 8000

def load_8k(path, seg=None):
    wav, sr = torchaudio.load(path)
    wav = torchaudio.functional.resample(wav, sr, SR).mean(0)
    if seg:
        if wav.shape[0] < seg:
            wav = torch.nn.functional.pad(wav, (0, seg - wav.shape[0]))
        else:
            s = random.randint(0, wav.shape[0] - seg)
            wav = wav[s:s+seg]
    return wav

def make_mix(n_spk=4, seg=SR*3):
    spks = random.sample(list(by_spk), n_spk)
    srcs = []
    for s in spks:
        w = load_8k(random.choice(by_spk[s]), seg)
        w = w / (w.abs().max() + 1e-8) * random.uniform(0.5, 1.0)
        srcs.append(w)
    srcs = torch.stack(srcs)                 
    mix = srcs.sum(0)
    scale = 0.9 / max(mix.abs().max().item(), 1e-8)
    if scale < 1:
        mix, srcs = mix * scale, srcs * scale
    return mix, srcs

In [6]:
from speechbrain.inference.separation import SepformerSeparation as Sep
m3 = Sep.from_hparams("speechbrain/sepformer-wsj03mix", savedir="sf3",
                      run_opts={"device": "cuda"})

mix, srcs = make_mix(3, seg=SR*4)
est = m3.separate_batch(mix.unsqueeze(0).cuda())      
torchaudio.save("mix3.wav", mix.unsqueeze(0), SR)
for i in range(3):
    torchaudio.save(f"est3_{i}.wav", est[0, :, i].unsqueeze(0).cpu(), SR)
print("saved — now LISTEN to all four files")

hyperparams.yaml: 0.00B [00:00, ?B/s]

masknet.ckpt:   0%|          | 0.00/113M [00:00<?, ?B/s]

encoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

decoder.ckpt:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


saved — now LISTEN to all four files


In [7]:
m2 = Sep.from_hparams("speechbrain/sepformer-wsj02mix", savedir="sf2",
                      run_opts={"device": "cuda"})

def split_score(a, b):
    cos = torch.nn.functional.cosine_similarity(a, b, dim=0).abs().item()
    ra, rb = a.pow(2).mean().sqrt().item(), b.pow(2).mean().sqrt().item()
    balance = min(ra, rb) / (max(ra, rb) + 1e-8)
    return (1 - cos) * balance          

def separate4_cascade(mix):             
    tri = m3.separate_batch(mix.unsqueeze(0))[0]             
    cands = []
    for i in range(3):
        duo = m2.separate_batch(tri[:, i].unsqueeze(0).T if False else tri[:, i].unsqueeze(0))[0]
        cands.append((split_score(duo[:, 0], duo[:, 1]), i, duo))
    _, k, duo = max(cands)
    keep = [tri[:, i] for i in range(3) if i != k]
    L = min(min(t.shape[0] for t in keep), duo.shape[0])
    return torch.stack([t[:L] for t in keep] + [duo[:L, 0], duo[:L, 1]])   

mix4, srcs4 = make_mix(4, seg=SR*4)
est4 = separate4_cascade(mix4.cuda())
print(est4.shape)                       

hyperparams.yaml: 0.00B [00:00, ?B/s]

masknet.ckpt:   0%|          | 0.00/113M [00:00<?, ?B/s]

encoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

decoder.ckpt:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


torch.Size([4, 32000])


In [8]:
from asteroid.losses import PITLossWrapper, pairwise_neg_sisdr
pit = PITLossWrapper(pairwise_neg_sisdr, pit_from="pw_mtx")

def sisdri(est, srcs, mix):
    L = min(est.shape[1], srcs.shape[1], mix.shape[0])
    est, srcs, mix = est[:, :L], srcs[:, :L], mix[:L]
    best = -pit(est.unsqueeze(0).cpu(), srcs.unsqueeze(0)).item()
    base = -pit(mix.repeat(len(srcs), 1).unsqueeze(0), srcs.unsqueeze(0)).item()
    return best - base

scores = []
for _ in range(10):
    mix4, srcs4 = make_mix(4, seg=SR*4)
    scores.append(sisdri(separate4_cascade(mix4.cuda()), srcs4, mix4))
print("cascade SI-SDRi over 10 mixes:", sum(scores)/len(scores), "dB")

/usr/local/lib/python3.12/dist-packages/asteroid/losses/pit_wrapper.py:314: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch_indices = torch.tensor([linear_sum_assignment(pwl)[1] for pwl in pwl_copy]).to(


cascade SI-SDRi over 10 mixes: 10.349899935722352 dB


In [9]:
m4 = Sep.from_hparams("speechbrain/sepformer-wsj03mix", savedir="sf4",
                      run_opts={"device": "cuda"})
print(m4.mods.masknet)        

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Dual_Path_Model(
  (norm): GroupNorm(1, 256, eps=1e-08, affine=True)
  (conv1d): Conv1d(256, 256, kernel_size=(1,), stride=(1,), bias=False)
  (dual_mdl): ModuleList(
    (0-1): 2 x Dual_Computation_Block(
      (intra_mdl): SBTransformerBlock(
        (mdl): TransformerEncoder(
          (layers): ModuleList(
            (0-7): 8 x TransformerEncoderLayer(
              (self_att): MultiheadAttention(
                (att): MultiheadAttention(
                  (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
                )
              )
              (pos_ffn): PositionalwiseFeedForward(
                (ffn): Sequential(
                  (0): Linear(in_features=256, out_features=1024, bias=True)
                  (1): ReLU()
                  (2): Dropout(p=0, inplace=False)
                  (3): Linear(in_features=1024, out_features=256, bias=True)
                )
              )
              (norm1): LayerNorm(
                (no

In [10]:
import torch.nn as nn
mn = m4.mods.masknet
print("num_spks before:", mn.num_spks)         

old = mn.conv2d
new = nn.Conv2d(256, 256*4, kernel_size=1)
with torch.no_grad():
    new.weight[:768] = old.weight              
    new.bias[:768]   = old.bias                 
mn.conv2d = new.cuda()
mn.num_spks = 4

masks = mn(m4.mods.encoder(torch.randn(1, SR*3).cuda()))
print("masks:", masks.shape)                    

num_spks before: 3
masks: torch.Size([4, 1, 256, 2999])


In [12]:
enc, mn, dec = m4.mods.encoder, m4.mods.masknet, m4.mods.decoder
m4.mods.train()
opt = torch.optim.Adam(m4.mods.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

def forward4(mix):                               
    w = enc(mix)
    masks = mn(w)
    return torch.stack([dec(w * masks[i]) for i in range(4)], dim=1)

for step in range(1, 20001):
    mix, srcs = make_mix(4)
    mix, srcs = mix.unsqueeze(0).cuda(), srcs.unsqueeze(0).cuda()
    with torch.amp.autocast("cuda"):
        est = forward4(mix)
    L = min(est.shape[-1], srcs.shape[-1])
    loss = pit(est[..., :L].float(), srcs[..., :L])
    opt.zero_grad(); scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(m4.mods.parameters(), 5)
    scaler.step(opt); scaler.update()
    if step == 10000:
        for g in opt.param_groups: g["lr"] = 5e-5
    if step % 100 == 0:
        print(step, "train SI-SNR:", round(-loss.item(), 2), "dB", flush=True)
    if step % 1000 == 0:
        torch.save(m4.mods.state_dict(), "sepformer4.ckpt")

100 train SI-SNR: 2.26 dB
200 train SI-SNR: 1.82 dB
300 train SI-SNR: 0.89 dB
400 train SI-SNR: 2.71 dB
500 train SI-SNR: 4.05 dB
600 train SI-SNR: 4.79 dB
700 train SI-SNR: -0.46 dB
800 train SI-SNR: 0.31 dB
900 train SI-SNR: -0.21 dB
1000 train SI-SNR: 2.77 dB
1100 train SI-SNR: 2.6 dB
1200 train SI-SNR: 2.96 dB
1300 train SI-SNR: 6.95 dB
1400 train SI-SNR: 4.98 dB
1500 train SI-SNR: 4.09 dB
1600 train SI-SNR: 5.1 dB
1700 train SI-SNR: 5.54 dB
1800 train SI-SNR: 5.69 dB
1900 train SI-SNR: 3.05 dB
2000 train SI-SNR: 3.63 dB
2100 train SI-SNR: 6.74 dB
2200 train SI-SNR: 5.46 dB
2300 train SI-SNR: 3.78 dB
2400 train SI-SNR: -2.51 dB
2500 train SI-SNR: -1.13 dB
2600 train SI-SNR: 4.11 dB
2700 train SI-SNR: 7.3 dB
2800 train SI-SNR: 6.61 dB
2900 train SI-SNR: 4.17 dB
3000 train SI-SNR: 5.27 dB
3100 train SI-SNR: 6.11 dB
3200 train SI-SNR: 4.13 dB
3300 train SI-SNR: 4.46 dB
3400 train SI-SNR: 7.77 dB
3500 train SI-SNR: 5.27 dB
3600 train SI-SNR: 4.46 dB
3700 train SI-SNR: -2.78 dB
3800 tra

In [13]:
import random
from asteroid.losses import PITLossWrapper, pairwise_neg_sisdr
pit = PITLossWrapper(pairwise_neg_sisdr, pit_from="pw_mtx")

def sisdri(est, srcs, mix):
    est = est.detach().cpu()
    L = min(est.shape[1], srcs.shape[1], mix.shape[0])
    est, srcs, mix = est[:, :L], srcs[:, :L], mix[:L]
    best = -pit(est.unsqueeze(0), srcs.unsqueeze(0)).item()
    base = -pit(mix.repeat(len(srcs), 1).unsqueeze(0), srcs.unsqueeze(0)).item()
    return best - base

m4.mods.eval()
def separate4_ft(mix):
    with torch.no_grad():
        return forward4(mix.unsqueeze(0).cuda())[0]

random.seed(7)
test_set = [make_mix(4, seg=SR*4) for _ in range(10)]

ft = [sisdri(separate4_ft(m), s, m) for m, s in test_set]
print("FINE-TUNED SI-SDRi:", round(sum(ft)/len(ft), 2), "dB")

try:
    cs = [sisdri(separate4_cascade(m.cuda()), s, m) for m, s in test_set]
    print("CASCADE    SI-SDRi:", round(sum(cs)/len(cs), 2), "dB")
except NameError:
    print("cascade not in this session — rerun the m3/m2 + cascade cells, then rerun this cell")

out = separate4_ft(test_set[0][0]).cpu()
torchaudio.save("demo_mix4.wav", test_set[0][0].unsqueeze(0), SR)
for i in range(4):
    torchaudio.save(f"demo_finetuned_out{i+1}.wav", out[i].unsqueeze(0), SR)

FINE-TUNED SI-SDRi: 9.6 dB
CASCADE    SI-SDRi: 7.22 dB
